# 02 — Regimes × Morfologia: Kruskal-Wallis (Cap. 6.2)Testa se os indicadores de purificação diferem significativamente entre regimes iconocráticos.**H₀:** A distribuição de cada indicador é igual entre os três regimes.**H₁:** Ao menos um regime difere.

In [ ]:
import pandas as pd, numpy as npfrom scipy import statsimport scikit_posthocs as spimport seaborn as sns, matplotlib.pyplot as pltsns.set_theme(style="whitegrid", font_scale=1.1)df = pd.read_csv('../data/processed/corpus_dataset.csv')INDICATORS = ['desincorporacao','rigidez_postural','dessexualizacao','uniformizacao_facial','heraldizacao','enquadramento_arquitetonico','apagamento_narrativo','monocromatizacao','serialidade','inscricao_estatal']INDICATOR_LABELS = {'desincorporacao':'Desincorporação','rigidez_postural':'Rigidez postural','dessexualizacao':'Dessexualização','uniformizacao_facial':'Uniformização facial','heraldizacao':'Heraldicização','enquadramento_arquitetonico':'Enquadramento arq.','apagamento_narrativo':'Apagamento narrativo','monocromatizacao':'Monocromatização','serialidade':'Serialidade','inscricao_estatal':'Inscrição estatal'}regimes = ['fundacional','normativo','militar']groups = {r: df[df.regime_iconocratico == r] for r in regimes}print(f"N por regime: {', '.join(f'{r}={len(g)}' for r, g in groups.items())}")print(f"\n{'='*70}")print(f"{'Indicador':<25s} {'H':>8s} {'p-value':>10s} {'Sig.':>6s} {'η²':>8s}")print(f"{'='*70}")results = []for ind in INDICATORS:    samples = [g[ind].dropna().values for g in groups.values()]    H, p = stats.kruskal(*samples)    n = sum(len(s) for s in samples); eta2 = (H - len(regimes) + 1) / (n - len(regimes))    sig = '***' if p < 0.001 else '**' if p < 0.01 else '*' if p < 0.05 else 'ns'    results.append({'indicator': ind, 'H': H, 'p': p, 'sig': sig, 'eta2': eta2})    print(f"{INDICATOR_LABELS[ind]:<25s} {H:8.2f} {p:10.4f} {sig:>6s} {eta2:8.3f}")print(f"{'='*70}")print(f"\n{sum(1 for r in results if r['p'] < 0.05)}/10 indicadores com diferença significativa (p < 0.05)")

## 2.2 Post-hoc Dunn (Bonferroni) para indicadores significativos

In [ ]:
sig_indicators = [r['indicator'] for r in results if r['p'] < 0.05]for ind in sig_indicators:    print(f"\n{'='*50}"); print(f"  Dunn post-hoc: {INDICATOR_LABELS[ind]}"); print(f"{'='*50}")    dunn = sp.posthoc_dunn(df, val_col=ind, group_col='regime_iconocratico', p_adjust='bonferroni')    print(dunn.round(4))print(f"\n{'='*50}"); print(f"  Kruskal-Wallis: Score composto de ENDURECIMENTO"); print(f"{'='*50}")H, p = stats.kruskal(*[groups[r]['purificacao_composto'].values for r in regimes])print(f"H = {H:.2f}, p = {p:.6f}")if p < 0.05:    dunn_comp = sp.posthoc_dunn(df, val_col='purificacao_composto', group_col='regime_iconocratico', p_adjust='bonferroni')    print("\nDunn post-hoc (Bonferroni):"); print(dunn_comp.round(4))

## 2.3 Visualização: violinplots por regime

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(20, 8), sharey=True)palette = {'fundacional': '#e74c3c', 'normativo': '#3498db', 'militar': '#2c3e50'}for i, ind in enumerate(INDICATORS):    ax = axes[i // 5, i % 5]    sns.violinplot(data=df, x='regime_iconocratico', y=ind, order=regimes, palette=palette, ax=ax, inner='quartile', cut=0)    sig = [r['sig'] for r in results if r['indicator'] == ind][0]    ax.set_title(f"{INDICATOR_LABELS[ind]}\n(KW {sig})", fontsize=9)    ax.set_xlabel(''); ax.set_ylabel('' if i % 5 else 'Score (0-3)')    ax.set_xticklabels(['F','N','M'], fontsize=8)plt.suptitle('Distribuição dos 10 indicadores por regime', fontsize=14, y=1.02)plt.tight_layout(); plt.savefig('../data/processed/fig_06_violins.png', dpi=150, bbox_inches='tight'); plt.show()